In [9]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch

# Root folder where images live
TEST_ROOT = "../data/test"

# List all breed folders under data/test
breed_dirs = [
    d for d in os.listdir(TEST_ROOT)
    if os.path.isdir(os.path.join(TEST_ROOT, d))
]

print("Test breed folders:", breed_dirs)

# Build a simple DataFrame-like list from the filesystem
filepaths = []
labels = []

for breed in breed_dirs:
    folder = os.path.join(TEST_ROOT, breed)
    for fname in os.listdir(folder):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            filepaths.append(os.path.join(folder, fname))
            labels.append(breed)

print("Total test images found on disk:", len(filepaths))
print("First 5 paths:", filepaths[:5])
print("First 5 labels:", labels[:5])

Test breed folders: ['Yorkie', 'Greyhound', 'Corgi', 'Mex Hairless', 'Boston Terrier', 'Blenheim', 'Scotch Terrier', 'Rottweiler', 'Elk Hound', 'Schnauzer', 'Bull Mastiff', 'Pomeranian', 'Pekinese', 'Pit Bull', 'Basenji', 'French Bulldog', 'Coyote', 'Komondor', 'Great Perenees', 'Basset', 'Great Dane', 'Dhole', 'Lhasa', 'Labrador', 'Chinese Crested', 'Cocker', 'Chihuahua', 'Shiba Inu', 'Newfoundland', 'Dalmation', 'Bull Terrier', 'Pug', 'Afghan', 'Cairn', 'Beagle', 'Cockapoo', 'Vizsla', 'Collie', 'Irish Wolfhound', 'American Spaniel', 'Maltese', 'African Wild Dog', 'Doberman', 'Shih-Tzu', 'German Sheperd', 'Labradoodle', 'Groenendael', 'Border Collie', 'Shar_Pei', 'Bearded Collie', 'Bermaise', 'Saint Bernard', 'Rhodesian', 'Siberian Husky', 'Japanese Spaniel', 'Borzoi', 'Chow', 'Airedale', 'Dingo', 'American Hairless', 'Irish Spaniel', 'Poodle', 'Malinois', 'Bichon Frise', 'Golden Retriever', 'Clumber', 'Boxer', 'Bulldog', 'Bloodhound', 'Bluetick']
Total test images found on disk: 700


In [4]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE      = 224

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

breeds = sorted(set(labels))
label2idx = {breed: idx for idx, breed in enumerate(breeds)}
print("Breeds:", breeds)
print("Number of breeds:", len(breeds))

class SimpleDogDataset(Dataset):
    def __init__(self, filepaths, labels, label2idx, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.label2idx = label2idx
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        label_name = self.labels[idx]
        img = Image.open(path).convert("RGB")
        label = self.label2idx[label_name]
        if self.transform:
            img = self.transform(img)
        return img, label

# For now, treat all test images as "train" for Sprint 3
train_dataset = SimpleDogDataset(filepaths, labels, label2idx, transform=test_transforms)
val_dataset   = SimpleDogDataset(filepaths, labels, label2idx, transform=test_transforms)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

images, labels_batch = next(iter(train_loader))
print("Sanity check batch:")
print(" Image batch shape:", images.shape)
print(" Label batch shape:", labels_batch.shape)

Breeds: ['Afghan', 'African Wild Dog', 'Airedale', 'American Hairless', 'American Spaniel', 'Basenji', 'Basset', 'Beagle', 'Bearded Collie', 'Bermaise', 'Bichon Frise', 'Blenheim', 'Bloodhound', 'Bluetick', 'Border Collie', 'Borzoi', 'Boston Terrier', 'Boxer', 'Bull Mastiff', 'Bull Terrier', 'Bulldog', 'Cairn', 'Chihuahua', 'Chinese Crested', 'Chow', 'Clumber', 'Cockapoo', 'Cocker', 'Collie', 'Corgi', 'Coyote', 'Dalmation', 'Dhole', 'Dingo', 'Doberman', 'Elk Hound', 'French Bulldog', 'German Sheperd', 'Golden Retriever', 'Great Dane', 'Great Perenees', 'Greyhound', 'Groenendael', 'Irish Spaniel', 'Irish Wolfhound', 'Japanese Spaniel', 'Komondor', 'Labradoodle', 'Labrador', 'Lhasa', 'Malinois', 'Maltese', 'Mex Hairless', 'Newfoundland', 'Pekinese', 'Pit Bull', 'Pomeranian', 'Poodle', 'Pug', 'Rhodesian', 'Rottweiler', 'Saint Bernard', 'Schnauzer', 'Scotch Terrier', 'Shar_Pei', 'Shiba Inu', 'Shih-Tzu', 'Siberian Husky', 'Vizsla', 'Yorkie']
Number of breeds: 70
Train batches: 22
Val batche

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

num_classes = len(breeds)
print("Number of breeds:", num_classes)

mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

for param in mobilenet.features.parameters():
    param.requires_grad = False

in_feats = mobilenet.classifier[1].in_features
mobilenet.classifier[1] = nn.Linear(in_feats, num_classes)

model = mobilenet.to(device)
print("Model ready")

criterion = nn.CrossEntropyLoss()
learning_rate = 1e-3
weight_decay = 1e-4

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

num_epochs = 10  
print("Training for", num_epochs, "epochs")

Using device: cpu
Number of breeds: 70
Model ready
Training for 10 epochs


In [8]:
train_losses = []
valid_losses = []

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0

    for images, labels_batch in train_loader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)

    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    running_valid_loss = 0.0

    with torch.no_grad():
        for images, labels_batch in val_loader:
            images = images.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels_batch)

            running_valid_loss += loss.item() * images.size(0)

    epoch_valid_loss = running_valid_loss / len(val_loader.dataset)
    valid_losses.append(epoch_valid_loss)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Valid Loss: {epoch_valid_loss:.4f}"
    )

Epoch 1/10 | Train Loss: 3.9624 | Valid Loss: 2.5878
Epoch 2/10 | Train Loss: 2.1868 | Valid Loss: 1.3523
Epoch 3/10 | Train Loss: 1.2828 | Valid Loss: 0.7965
Epoch 4/10 | Train Loss: 0.8229 | Valid Loss: 0.5232
Epoch 5/10 | Train Loss: 0.6350 | Valid Loss: 0.3972
Epoch 6/10 | Train Loss: 0.5042 | Valid Loss: 0.2976
Epoch 7/10 | Train Loss: 0.3800 | Valid Loss: 0.2584
Epoch 8/10 | Train Loss: 0.3321 | Valid Loss: 0.1960
Epoch 9/10 | Train Loss: 0.2774 | Valid Loss: 0.1704
Epoch 10/10 | Train Loss: 0.2346 | Valid Loss: 0.1491
